# Bootstrap run analysis

Crawls every training run under `MyDrive/reinforce-tactics/benchmarks/bootstrap/` and writes a single consolidated summary of each run: settings, enabled units, per-stage progression, and key outcomes.

**Outputs**
- `runs_summary.csv` — one row per run, top-level settings + curriculum reach.
- `runs_per_stage.csv` — one row per (run, stage), detailed stage configuration + final eval.
- `runs_detail.json` — full nested structure for programmatic re-analysis.

**Usage**
1. Run all cells in order.
2. Edit `DRIVE_BOOTSTRAP_ROOT` in the config cell if your folder layout differs.
3. The three output files land alongside the run folders so they're easy to download or share.

## Setup

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab; using local paths.')

In [ ]:
# Path to the bootstrap runs folder. Adjust if your layout differs.
DRIVE_BOOTSTRAP_ROOT = '/content/drive/MyDrive/reinforce-tactics/benchmarks/bootstrap'
OUTPUT_DIR = DRIVE_BOOTSTRAP_ROOT  # where to write the consolidated files

# Default unit roster when env.enabled_units is null (per
# reinforcetactics/game/llm_bot.py and config.py defaults).
ALL_UNITS = ['W', 'M', 'C', 'A', 'K', 'R', 'S', 'B']

RUN_ID_RE = re.compile(r'^\d{8}_\d{6}$')

## Discovery

In [ ]:
def find_runs(root: str) -> List[Path]:
    """Return run folders sorted oldest-first by timestamp prefix."""
    root_p = Path(root)
    if not root_p.exists():
        raise FileNotFoundError(f'Bootstrap root not found: {root}')
    runs = [p for p in root_p.iterdir() if p.is_dir() and RUN_ID_RE.match(p.name)]
    return sorted(runs, key=lambda p: p.name)


def find_config_yaml(run_dir: Path) -> Optional[Path]:
    """Locate the v*.yaml config stored alongside the run."""
    candidates = list(run_dir.glob('v*.yaml')) + list(run_dir.glob('*.yaml'))
    return candidates[0] if candidates else None


runs = find_runs(DRIVE_BOOTSTRAP_ROOT)
print(f'Found {len(runs)} run folder(s):')
for r in runs:
    print(f'  {r.name}')

## Parsing helpers

In [ ]:
def safe_get(d: Optional[dict], *keys: str, default: Any = None) -> Any:
    """Walk nested dicts, returning default on any missing key."""
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def normalise_enabled_units(value: Any) -> List[str]:
    """None / missing -> all 8 default units. Explicit list -> sorted copy."""
    if value is None:
        return list(ALL_UNITS)
    if isinstance(value, list):
        return sorted(set(value))
    return [str(value)]


def ent_coef_summary(ent: Any) -> Dict[str, Any]:
    """Flatten an ent_coef block (scalar or {start,end,schedule}) to fields."""
    if isinstance(ent, dict):
        return {
            'ent_start': ent.get('start'),
            'ent_end': ent.get('end'),
            'ent_schedule': ent.get('schedule', 'linear'),
        }
    if ent is None:
        return {'ent_start': None, 'ent_end': None, 'ent_schedule': None}
    return {'ent_start': ent, 'ent_end': ent, 'ent_schedule': 'flat'}

In [ ]:
def parse_config(yaml_path: Path) -> Dict[str, Any]:
    """Extract the top-level + per-stage settings from a curriculum YAML."""
    with yaml_path.open() as f:
        cfg = yaml.safe_load(f)

    env_units = safe_get(cfg, 'env', 'enabled_units')
    enabled_units = normalise_enabled_units(env_units)
    enabled_units_explicit = env_units is not None

    reward = safe_get(cfg, 'env', 'reward_config', default={}) or {}
    ppo = cfg.get('ppo', {}) or {}
    eval_cfg = cfg.get('eval', {}) or {}

    stages = []
    for s in safe_get(cfg, 'curriculum', 'stages', default=[]) or []:
        stage_units = s.get('enabled_units')
        stages.append({
            'stage_name': s.get('name'),
            'map_file': s.get('map_file'),
            'opponent': s.get('opponent'),
            'opponent_kwargs': s.get('opponent_kwargs') or {},
            'promotion_win_rate': s.get('promotion_win_rate'),
            'patience': s.get('patience'),
            'max_timesteps': s.get('max_timesteps'),
            'max_turns': s.get('max_turns'),
            'n_eval_episodes': s.get('n_eval_episodes'),
            'enabled_units_override': (
                normalise_enabled_units(stage_units) if stage_units is not None else None
            ),
            **ent_coef_summary(s.get('ent_coef')),
        })

    return {
        'config_name': yaml_path.stem,
        'algorithm': cfg.get('algorithm'),
        'seed': cfg.get('seed'),
        'total_timesteps_planned': cfg.get('total_timesteps'),
        'n_envs': safe_get(cfg, 'env', 'n_envs'),
        'max_steps_env': safe_get(cfg, 'env', 'max_steps'),
        'max_turns_env': safe_get(cfg, 'env', 'max_turns'),
        'max_actions_per_turn': safe_get(cfg, 'env', 'max_actions_per_turn'),
        'action_space_type': safe_get(cfg, 'env', 'action_space_type'),
        'enabled_units': enabled_units,
        'enabled_units_count': len(enabled_units),
        'enabled_units_explicit': enabled_units_explicit,
        # Hyperparameters
        'learning_rate': ppo.get('learning_rate'),
        'n_steps': ppo.get('n_steps'),
        'batch_size': ppo.get('batch_size'),
        'n_epochs': ppo.get('n_epochs'),
        'gamma': ppo.get('gamma'),
        'gae_lambda': ppo.get('gae_lambda'),
        'clip_range': ppo.get('clip_range'),
        'ent_coef_global': ppo.get('ent_coef'),
        'vf_coef': ppo.get('vf_coef'),
        'max_grad_norm': ppo.get('max_grad_norm'),
        'features_extractor_class': safe_get(
            ppo, 'policy_kwargs', 'features_extractor_class'
        ),
        'net_arch_pi': safe_get(ppo, 'policy_kwargs', 'net_arch', 'pi'),
        'net_arch_vf': safe_get(ppo, 'policy_kwargs', 'net_arch', 'vf'),
        # Reward landscape (the levers we tune across versions)
        'reward_win': reward.get('win'),
        'reward_loss': reward.get('loss'),
        'reward_draw': reward.get('draw'),
        'reward_turn_penalty': reward.get('turn_penalty'),
        'reward_win_speed_bonus': reward.get('win_speed_bonus'),
        'reward_win_by_hq': reward.get('win_by_hq_capture'),
        'reward_win_by_elim': reward.get('win_by_elimination'),
        'reward_building_capture': reward.get('building_capture'),
        'reward_hq_capture': reward.get('hq_capture'),
        'reward_tower_capture': reward.get('tower_capture'),
        'reward_invalid_action': reward.get('invalid_action'),
        'reward_enemy_owned_capture': reward.get('enemy_owned_capture'),
        'reward_enemy_neutral_capture': reward.get('enemy_neutral_capture'),
        # Eval cadence
        'eval_freq': eval_cfg.get('eval_freq'),
        'eval_n_episodes_default': eval_cfg.get('n_eval_episodes'),
        # Curriculum structure
        'stage_count': len(stages),
        'stages': stages,
    }

In [ ]:
def parse_bootstrap_csv(csv_path: Path) -> pd.DataFrame:
    """Load bootstrap_results.csv, cleaning the column headers (Drive exports
    sometimes inject \\_ escapes into header names)."""
    df = pd.read_csv(csv_path)
    df.columns = [c.replace('\\_', '_').strip() for c in df.columns]
    return df


def stage_outcome(stage_df: pd.DataFrame, threshold: float, patience: int) -> str:
    """Classify how a stage ended.

    A stage is `cleared` if its last `patience` evals all met threshold (the
    curriculum's promotion gate). Otherwise `stalled`. Empty -> `unknown`.
    """
    if stage_df.empty or threshold is None or patience is None:
        return 'unknown'
    tail = stage_df.tail(int(patience))
    if len(tail) < int(patience):
        return 'in_progress'
    return 'cleared' if (tail['win_rate'] >= float(threshold)).all() else 'stalled'


def per_stage_summary(
    df: pd.DataFrame, cfg_stages: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    seen_names = set()
    for stage_cfg in cfg_stages:
        name = stage_cfg['stage_name']
        seen_names.add(name)
        sub = df[df['stage'] == name].sort_values('timesteps')
        if sub.empty:
            out.append({
                **stage_cfg,
                'evals_run': 0,
                'first_step': None,
                'last_step': None,
                'steps_in_stage': 0,
                'final_win_rate': None,
                'peak_win_rate': None,
                'final_avg_reward': None,
                'final_avg_turns': None,
                'outcome': 'not_started',
            })
            continue
        first = int(sub['timesteps'].iloc[0])
        last = int(sub['timesteps'].iloc[-1])
        out.append({
            **stage_cfg,
            'evals_run': int(len(sub)),
            'first_step': first,
            'last_step': last,
            'steps_in_stage': last - first,
            'final_win_rate': float(sub['win_rate'].iloc[-1]),
            'peak_win_rate': float(sub['win_rate'].max()),
            'final_avg_reward': float(sub['avg_reward'].iloc[-1]),
            'final_avg_turns': float(sub['avg_turns'].iloc[-1]),
            'outcome': stage_outcome(
                sub,
                stage_cfg.get('promotion_win_rate'),
                stage_cfg.get('patience'),
            ),
        })
    # Any stages that appear in the CSV but not in the config (shouldn't happen,
    # but worth surfacing if it does).
    extras = sorted(set(df['stage'].dropna().unique()) - seen_names)
    for name in extras:
        sub = df[df['stage'] == name].sort_values('timesteps')
        out.append({
            'stage_name': name,
            'opponent': None,
            'opponent_kwargs': {},
            'promotion_win_rate': None,
            'patience': None,
            'max_timesteps': None,
            'ent_start': None,
            'ent_end': None,
            'ent_schedule': None,
            'evals_run': int(len(sub)),
            'first_step': int(sub['timesteps'].min()),
            'last_step': int(sub['timesteps'].max()),
            'steps_in_stage': int(sub['timesteps'].max() - sub['timesteps'].min()),
            'final_win_rate': float(sub['win_rate'].iloc[-1]),
            'peak_win_rate': float(sub['win_rate'].max()),
            'outcome': 'unknown_in_config',
        })
    return out

In [ ]:
def find_meta_path(run_dir: Path) -> Optional[Path]:
    """Locate any per-stage config.json the training pipeline writes.

    reinforcetactics/utils/run_config.py writes one config.json per stage
    subfolder with shared git/library/hardware metadata for the run. We
    pick the alphabetically first match for determinism.
    """
    candidates = sorted(run_dir.glob('*/config.json'))
    return candidates[0] if candidates else None


def extract_run_meta(meta_path: Path) -> Dict[str, Any]:
    """Pull git, library, and hardware fields out of a stage config.json."""
    with meta_path.open() as f:
        meta_cfg = json.load(f)
    meta = meta_cfg.get('meta', {}) or {}
    git = meta.get('git', {}) or {}
    libs = meta.get('libraries', {}) or {}
    hw = meta.get('hardware', {}) or {}
    return {
        'meta_source_stage': meta_path.parent.name,
        'meta_created_at': meta.get('created_at'),
        'git_commit': git.get('commit'),
        'git_short': git.get('short'),
        'git_dirty': git.get('dirty'),
        'lib_reinforcetactics': libs.get('reinforcetactics'),
        'lib_torch': libs.get('torch'),
        'lib_numpy': libs.get('numpy'),
        'lib_gymnasium': libs.get('gymnasium'),
        'lib_stable_baselines3': libs.get('stable_baselines3'),
        'lib_sb3_contrib': libs.get('sb3_contrib'),
        'hw_platform': hw.get('platform'),
        'hw_python': hw.get('python'),
        'hw_cpu_count': hw.get('cpu_count'),
        'hw_ram_gb': hw.get('ram_gb'),
        'hw_gpu_name': hw.get('gpu_name'),
        'hw_gpu_count': hw.get('gpu_count'),
        'hw_torch_cuda_available': hw.get('torch_cuda_available'),
        'hw_torch_cuda_version': hw.get('torch_cuda_version'),
    }


## Per-run aggregation

In [ ]:
def analyse_run(run_dir: Path) -> Dict[str, Any]:
    """Return a flat-ish dict summarising one run."""
    run_id = run_dir.name
    yaml_path = find_config_yaml(run_dir)
    bootstrap_csv = run_dir / 'bootstrap_results.csv'
    has_yaml = yaml_path is not None
    has_csv = bootstrap_csv.exists()

    record: Dict[str, Any] = {
        'run_id': run_id,
        'run_path': str(run_dir),
        'has_config_yaml': has_yaml,
        'has_bootstrap_csv': has_csv,
        'config_path': str(yaml_path) if yaml_path else None,
    }

    # Per-stage config.json captures git commit + library versions +
    # hardware (written by reinforcetactics/utils/run_config.py). Read
    # whichever stage was written first; all stages of a run share the
    # same fields. Older runs may not have it.
    meta_path = find_meta_path(run_dir)
    if meta_path is not None:
        record.update(extract_run_meta(meta_path))
        record['has_run_meta'] = True
    else:
        record['has_run_meta'] = False

    if not has_yaml:
        record['error'] = 'no_yaml_config_found'
        return record

    cfg = parse_config(yaml_path)
    record.update({k: v for k, v in cfg.items() if k != 'stages'})
    cfg_stages = cfg['stages']
    record['enabled_units_str'] = ','.join(cfg['enabled_units'])

    if not has_csv:
        record['error'] = 'no_bootstrap_csv'
        record['stages'] = cfg_stages
        return record

    df = parse_bootstrap_csv(bootstrap_csv)
    stage_rows = per_stage_summary(df, cfg_stages)
    record['stages'] = stage_rows

    cleared = [s for s in stage_rows if s['outcome'] == 'cleared']
    stalled = [s for s in stage_rows if s['outcome'] == 'stalled']
    last_active = next(
        (s for s in reversed(stage_rows) if s['outcome'] not in ('not_started',)),
        None,
    )

    record.update({
        'total_evals_logged': int(len(df)),
        'total_env_steps': int(df['timesteps'].max()) if not df.empty else 0,
        'stages_cleared': len(cleared),
        'stages_stalled': len(stalled),
        'final_stage': last_active['stage_name'] if last_active else None,
        'final_stage_outcome': last_active['outcome'] if last_active else None,
        'final_stage_peak_wr': last_active['peak_win_rate'] if last_active else None,
        'final_stage_final_wr': last_active['final_win_rate'] if last_active else None,
        'completed_curriculum': len(cleared) == len(cfg_stages),
    })
    return record


all_records = [analyse_run(r) for r in runs]
print(f'Analysed {len(all_records)} run(s).')

## Build consolidated tables

In [ ]:
# Top-level: one row per run.
summary_columns_drop = ['stages']  # nested, handled separately
summary_rows = [{k: v for k, v in r.items() if k not in summary_columns_drop} for r in all_records]
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values('run_id').reset_index(drop=True)

# Per-stage: one row per (run, stage).
per_stage_rows: List[Dict[str, Any]] = []
for r in all_records:
    for s in r.get('stages') or []:
        per_stage_rows.append({
            'run_id': r['run_id'],
            'config_name': r.get('config_name'),
            **{k: v for k, v in s.items() if k != 'opponent_kwargs'},
            'opponent_max_actions': (s.get('opponent_kwargs') or {}).get('max_actions'),
            'opponent_p_hard': (s.get('opponent_kwargs') or {}).get('p_hard'),
            'opponent_easy': (s.get('opponent_kwargs') or {}).get('easy'),
            'opponent_hard': (s.get('opponent_kwargs') or {}).get('hard'),
            'enabled_units_override_str': (
                ','.join(s['enabled_units_override'])
                if s.get('enabled_units_override')
                else None
            ),
        })
per_stage_df = pd.DataFrame(per_stage_rows)

summary_df.head(20)

In [ ]:
per_stage_df.head(30)

## Save

In [ ]:
out_root = Path(OUTPUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)

summary_path = out_root / 'runs_summary.csv'
per_stage_path = out_root / 'runs_per_stage.csv'
detail_path = out_root / 'runs_detail.json'

summary_df.to_csv(summary_path, index=False)
per_stage_df.to_csv(per_stage_path, index=False)
with detail_path.open('w') as f:
    json.dump(all_records, f, indent=2, default=str)

print(f'Wrote:\n  {summary_path}\n  {per_stage_path}\n  {detail_path}')

## Quick views

Curriculum reach (`stages_cleared / stage_count`) by run, plus a compact view of which units each run had enabled. Plot only if matplotlib is available.

In [ ]:
view = summary_df.copy()
view['reach'] = (
    view['stages_cleared'].fillna(0).astype(int).astype(str)
    + '/'
    + view['stage_count'].fillna(0).astype(int).astype(str)
)
view[[
    'run_id',
    'config_name',
    'reach',
    'final_stage',
    'final_stage_outcome',
    'final_stage_peak_wr',
    'total_env_steps',
    'enabled_units_str',
    'enabled_units_explicit',
    'reward_turn_penalty',
    'seed',
    'git_short',
    'git_dirty',
    'lib_reinforcetactics',
    'lib_torch',
    'hw_gpu_name',
]]

In [ ]:
try:
    import matplotlib.pyplot as plt

    plot_df = summary_df.dropna(subset=['stages_cleared', 'stage_count']).copy()
    plot_df['stages_cleared'] = plot_df['stages_cleared'].astype(int)
    plot_df['stage_count'] = plot_df['stage_count'].astype(int)

    fig, ax = plt.subplots(figsize=(12, max(3, 0.4 * len(plot_df))))
    y = list(range(len(plot_df)))
    ax.barh(y, plot_df['stage_count'], color='lightgray', label='curriculum length')
    ax.barh(y, plot_df['stages_cleared'], color='tab:green', label='cleared')
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['run_id'] + '  (' + plot_df['config_name'].fillna('?') + ')')
    ax.invert_yaxis()
    ax.set_xlabel('Stages')
    ax.set_title('Curriculum reach per run')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib unavailable; skipping plot.')

In [ ]:
# Units enabled per run -- highlights any run that ran with a non-default roster.
unit_view = summary_df[['run_id', 'config_name', 'enabled_units_str', 'enabled_units_explicit', 'enabled_units_count']].copy()
unit_view['default_roster'] = ~unit_view['enabled_units_explicit'].fillna(False)
unit_view

In [ ]:
# Library/git fingerprint per run -- highlights any drift in versions
# or commits across the run set.
version_cols = [
    'run_id', 'config_name', 'git_short', 'git_dirty',
    'lib_reinforcetactics', 'lib_torch', 'lib_stable_baselines3',
    'lib_sb3_contrib', 'lib_gymnasium', 'lib_numpy',
    'hw_python', 'hw_gpu_name',
]
version_view = summary_df[
    [c for c in version_cols if c in summary_df.columns]
].copy()
version_view